# Analysis of League of Legends Dataset

**Name(s)**: Bruce

**Website Link**: (your website link)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import *

## Step 1: Introduction

In [2]:
path = Path('2025_LoL_esports_match_data_from_OraclesElixir.csv')
lol = pd.read_csv(path)
lol.head()

/var/folders/8z/t3klk_9d3sbf04zdg07sk1ch0000gn/T/ipykernel_32367/1291358343.py:2: DtypeWarning:

Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.



,gameid,datacompleteness,url,league,...,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,2.0,4.0,2.0
1,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,1.0,7.0,0.0
2,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,1.0,5.0,1.0
3,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,6.0,2.0,0.0
4,LOLTMNT03_179647,complete,NaN,LFL2,...,2.0,0.0,8.0,0.0


In [3]:
lol.shape

(120636, 165)

In [4]:
team_only = lol[lol['position'] == 'team'].copy()
team_only[team_only["atakhans"].isna()]["league"].value_counts()

league
LPL       1610
WLDs        24
LVP SL       8
          ... 
NLC          2
TCL          2
HLL          2
Name: count, Length: 14, dtype: int64

In [5]:
(lol.isna().mean() * 100).sort_values(ascending=False)

dragons (type unknown)     98.65
url                        91.87
monsterkillsenemyjungle    91.87
                           ...  
damagetochampions           0.00
ckpm                        0.00
damagetakenperminute        0.00
Length: 165, dtype: float64

How does the number of objectives secured affect the outcome of a match?

In [6]:
short = lol[['gameid', 'league', 'date', 'position', 'side', 'teamname', 'result', 'teamkills', 'teamdeaths', 'dragons', 'elders', 'heralds', 'void_grubs', 'barons', 'atakhans', 'damagetochampions', 'totalgold', 'monsterkills']].copy()
short.head()

,gameid,league,date,position,...,atakhans,damagetochampions,totalgold,monsterkills
0,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,top,...,NaN,20156,10668,0
1,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,jng,...,NaN,4963,7429,132
2,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,mid,...,NaN,13952,9032,0
3,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,bot,...,NaN,6898,9407,12
4,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,sup,...,NaN,4174,5719,0


In [7]:
short = short[short['position'] == 'team'].copy()
short

,gameid,league,date,position,...,atakhans,damagetochampions,totalgold,monsterkills
10,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,team,...,0.0,50143,42255,144
11,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,team,...,1.0,53681,53936,169
22,LOLTMNT06_96134,LFL2,2025-01-11 12:06:37,team,...,1.0,100112,64669,199
...,...,...,...,...,...,...,...,...,...
120623,LOLTMNT03_331511,DCup,2025-12-29 09:57:49,team,...,1.0,84235,76363,267
120634,LOLTMNT03_332179,DCup,2025-12-29 10:52:27,team,...,0.0,74670,51428,177
120635,LOLTMNT03_332179,DCup,2025-12-29 10:52:27,team,...,1.0,97445,64861,215


In [8]:
(short.isna().mean() * 100).sort_values(ascending=False)

atakhans        8.39
elders          8.13
gameid          0.00
                ... 
position        0.00
date            0.00
monsterkills    0.00
Length: 18, dtype: float64

In [9]:
short[short["atakhans"].isna()][["gameid", "teamname", 'league', "atakhans", "elders"]].head(20)

,gameid,teamname,league,atakhans,elders
58,11715-11715_game_1,Weibo Gaming,LPL,NaN,NaN
59,11715-11715_game_1,Oh My God,LPL,NaN,NaN
70,11715-11715_game_2,Oh My God,LPL,NaN,NaN
...,...,...,...,...,...
299,11717-11717_game_1,FunPlus Phoenix,LPL,NaN,NaN
310,11717-11717_game_2,Royal Never Give Up,LPL,NaN,NaN
311,11717-11717_game_2,FunPlus Phoenix,LPL,NaN,NaN


In [10]:
short[short['atakhans'].isna()]['league'].unique()

array(['LPL', 'LCKC', 'LVP SL', 'LCK', 'LIT', 'LFL', 'LJL', 'LTA S', 'CD',
       'NLC', 'TCL', 'HLL', 'NEXO', 'WLDs'], dtype=object)

In [11]:
short[
    (short["atakhans"].isna()) & (short["league"] != "LPL")
].drop_duplicates("gameid")[["league", "gameid", "date"]].sort_values("date")

,league,gameid,date
166,LCKC,LOLTMNT03_180652,2025-01-13 07:32:39
622,LVP SL,LOLTMNT03_182901,2025-01-15 19:44:50
2110,LCK,LOLTMNT03_188641,2025-01-22 08:04:43
...,...,...,...
110782,WLDs,12838-12838_game_2,2025-09-27 10:07:45
110794,WLDs,12838-12838_game_3,2025-09-27 11:07:55
110806,WLDs,12838-12838_game_4,2025-09-27 12:01:17


In [12]:
short[short['elders'].isna()]['league'].unique()

array(['LPL', 'WLDs'], dtype=object)

In [13]:
short[short['elders'].isna()]['league'].value_counts()

league
LPL     1610
WLDs      24
Name: count, dtype: int64

## Step 2: Data Cleaning and Exploratory Data Analysis

In [14]:
short_clean = short.drop(columns=["atakhans"])
short_clean = short_clean.dropna(subset=["elders"])

In [15]:
obj_cols = ["dragons", "barons", "void_grubs", 'heralds', "elders"]
short_clean["total_obj"] = short_clean[obj_cols].sum(axis=1)
short_clean = short_clean.reset_index(drop=True)
short_clean.head(8)

,gameid,league,date,position,...,damagetochampions,totalgold,monsterkills,total_obj
0,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,team,...,50143,42255,144,0.0
1,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,team,...,53681,53936,169,10.0
2,LOLTMNT06_96134,LFL2,2025-01-11 12:06:37,team,...,100112,64669,199,11.0
...,...,...,...,...,...,...,...,...,...
5,LOLTMNT06_95160,LFL2,2025-01-11 13:07:47,team,...,86703,62965,180,10.0
6,LOLTMNT03_178705,LFL2,2025-01-11 14:03:27,team,...,54173,40489,143,1.0
7,LOLTMNT03_178705,LFL2,2025-01-11 14:03:27,team,...,57648,51018,171,10.0


In [35]:
print(short_clean[[
    "league",
    "result",
    "position",
    'totalgold',
    "barons",
    "dragons",
    'heralds',
    'void_grubs',
    'total_obj'
]].head().to_markdown(index=False))

| league   |   result | position   |   totalgold |   barons |   dragons |   heralds |   void_grubs |   total_obj |
|:---------|---------:|:-----------|------------:|---------:|----------:|----------:|-------------:|------------:|
| LFL2     |        0 | team       |       42255 |        0 |         0 |         0 |            0 |           0 |
| LFL2     |        1 | team       |       53936 |        1 |         2 |         1 |            6 |          10 |
| LFL2     |        1 | team       |       64669 |        1 |         3 |         1 |            6 |          11 |
| LFL2     |        0 | team       |       50679 |        0 |         2 |         0 |            0 |           2 |
| LFL2     |        0 | team       |       51389 |        0 |         0 |         0 |            2 |           2 |


In [22]:
pivot = short_clean.pivot(index="gameid", columns="side", values=["total_obj", "result"])

higher_obj_won = (
    (pivot["total_obj"]["Blue"] > pivot["total_obj"]["Red"]) & (pivot["result"]["Blue"] == 1)
) | (
    (pivot["total_obj"]["Red"] > pivot["total_obj"]["Blue"]) & (pivot["result"]["Red"] == 1)
)

win_percent = float(higher_obj_won.mean())
win_percent

0.7708964919878736

In [23]:
objectives = ["dragons", "barons", "elders", "heralds", "void_grubs"]

winrates = {}

for obj in objectives:
    more_col = f"more_{obj}"
    short_clean[more_col] = (
        short_clean[obj] >
        short_clean.groupby("gameid")[obj].transform("mean")
    )
    winrates[obj] = short_clean.loc[short_clean[more_col], "result"].mean()

pd.Series(winrates)

dragons       0.83
barons        0.90
elders        0.87
heralds       0.66
void_grubs    0.60
dtype: float64

In [24]:
rows = []

for obj in objectives:
    
    more = short_clean[obj] > short_clean.groupby("gameid")[obj].transform("mean")
    less = short_clean[obj] < short_clean.groupby("gameid")[obj].transform("mean")
    
    win_more = short_clean.loc[more, "result"].mean()
    win_less = short_clean.loc[less, "result"].mean()
    
    rows.append({
        "objective": obj,
        "winrate_more": win_more,
        "winrate_less": win_less,
        "winrate_diff": win_more - win_less
    })

objective_df = pd.DataFrame(rows).sort_values("winrate_diff", ascending=False)

objective_df

,objective,winrate_more,winrate_less,winrate_diff
1,barons,0.90,0.10,0.81
2,elders,0.87,0.13,0.74
0,dragons,0.83,0.17,0.66
3,heralds,0.66,0.34,0.33
4,void_grubs,0.60,0.40,0.20


In [25]:
#Adding columns for differential in each objective
for m in objectives:
    short_clean[f"{m}_diff"] = (
        2 * short_clean[m] - short_clean.groupby("gameid")[m].transform("sum")
    )

In [26]:
short_clean["outcome"] = short_clean["result"].map({0: "Loss", 1: "Win"})

In [27]:
short_clean["total_obj_diff"] = (
    short_clean["total_obj"]
    - short_clean.groupby("gameid")["total_obj"].transform("mean")
) * 2

In [28]:
short_clean["early_obj_diff"] = (
    short_clean["heralds_diff"] + short_clean["void_grubs_diff"]
)

In [29]:
short_clean.head(5)

,gameid,league,date,position,...,void_grubs_diff,outcome,total_obj_diff,early_obj_diff
0,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,team,...,-6.0,Loss,-10.0,-7.0
1,LOLTMNT03_179647,LFL2,2025-01-11 11:11:24,team,...,6.0,Win,10.0,7.0
2,LOLTMNT06_96134,LFL2,2025-01-11 12:06:37,team,...,6.0,Win,9.0,7.0
3,LOLTMNT06_96134,LFL2,2025-01-11 12:06:37,team,...,-6.0,Loss,-9.0,-7.0
4,LOLTMNT06_95160,LFL2,2025-01-11 13:07:47,team,...,-2.0,Loss,-8.0,-3.0


In [30]:
fig1 = px.histogram(
    short_clean,
    x=short_clean["barons_diff"].abs(),
    title="Magnitude of Baron Advantage per Team",
    nbins=6,
    labels={"x": "Absolute Baron Difference"}
)

In [33]:
fig1.write_html("assets/baron_plot.html", include_plotlyjs="cdn")

In [31]:
fig2 = px.histogram(
    short_clean,
    x=short_clean["dragons_diff"].abs(),
    title="Magnitude of Dragon Advantage per Team",
    nbins=8,
    labels={"x": "Absolute Dragon Difference"}
)

In [34]:
fig2.write_html("assets/dragon_plot.html", include_plotlyjs="cdn")

In [28]:
px.histogram(
    short_clean,
    x="total_obj",
    title="Distribution of Total Objectives per Team",
    nbins=10
)

In [29]:
win_by_baron = short_clean.groupby("barons_diff")["result"].mean().reset_index()

px.bar(
    win_by_baron,
    x="barons_diff",
    y="result",
    title="Win Rate by Baron Difference",
    labels={"result": "Win Rate"}
)

In [30]:
px.box(
    short_clean,
    x="outcome",
    y="total_obj",
    title="Total Objectives for Winning vs Losing Teams"
)

In [31]:
win_by_diff = short_clean.groupby("total_obj_diff")["result"].mean().reset_index()

px.line(
    win_by_diff,
    x="total_obj_diff",
    y="result",
    title="Win Rate by Objective Difference",
    labels={"result": "Win Rate"}
)

In [32]:
pivot_table = (
    short_clean.groupby("barons_diff")
    .agg(
        games=("result", "count"),
        win_rate=("result", "mean")
    )
    .reset_index()
)

pivot_table = pivot_table[pivot_table["barons_diff"] >= 0]
pivot_table = pivot_table.sort_values("barons_diff")

pivot_table

,barons_diff,games,win_rate
4,0.0,4340,0.50
5,1.0,5998,0.91
6,2.0,962,0.88
7,3.0,102,0.86
8,4.0,4,0.75


## Step 3: Assessment of Missingness

In [35]:
team_only["atakhan_missing"] = team_only["atakhans"].isna()

In [ ]:
obs = team_only.groupby("league")["atakhan_missing"].mean()
observed_stat = obs.max() - obs.min()
observed_stat

1.0

In [37]:
perm_stats = []

for _ in range(1000):
    shuffled = np.random.permutation(team_only["atakhan_missing"])
    temp = team_only.copy()
    temp["shuffled"] = shuffled

    stat = (
        temp.groupby("league")["shuffled"].mean().max()
        - temp.groupby("league")["shuffled"].mean().min()
    )
    perm_stats.append(stat)

perm_stats = np.array(perm_stats)

p_value_league = (np.sum(perm_stats >= observed_stat) + 1) / (len(perm_stats) + 1)
p_value_league

0.000999000999000999

Missingness is dependent on league

In [38]:
obs_missing = team_only.loc[team_only["atakhan_missing"], "totalgold"].mean()
obs_not_missing = team_only.loc[~team_only["atakhan_missing"], "totalgold"].mean()

observed_stat_gold = abs(obs_missing - obs_not_missing)
observed_stat_gold

2.2114261536844424

In [39]:
perm_stats = []

for _ in range(1000):
    shuffled = np.random.permutation(team_only["atakhan_missing"])
    temp = team_only.copy()
    temp["shuffled"] = shuffled

    mean_missing = temp.loc[temp["shuffled"], "totalgold"].mean()
    mean_not_missing = temp.loc[~temp["shuffled"], "totalgold"].mean()

    stat = abs(mean_missing - mean_not_missing)
    perm_stats.append(stat)

perm_stats = np.array(perm_stats)

p_value_gold = (np.sum(perm_stats >= observed_stat_gold) + 1) / (len(perm_stats) + 1)
p_value_gold

0.995004995004995

Likely no relation between missingness and total gold.

In [40]:
league_missing_rate = (
    team_only.groupby("league")["atakhan_missing"]
    .mean()
    .reset_index(name="missing_rate")
)

league_missing_rate = league_missing_rate[
    league_missing_rate["missing_rate"] > 0
]

league_missing_rate = league_missing_rate.sort_values(
    "missing_rate", ascending=True
)

fig = px.bar(
    league_missing_rate,
    x="missing_rate",
    y="league",
    orientation="h",
    title="Proportion of Missing Atakhans by League",
    labels={
        "missing_rate": "Proportion Missing",
        "league": "League"
    }
)

fig.update_layout(
    yaxis=dict(automargin=True),
    height=400
)

fig.show()

## Step 4: Hypothesis Testing

Null: The difference in winrate advantage between Baron and Dragon objectives is due to random chance.



Alternative: Baron objective advantage leads to a larger winrate difference than Dragon objective advantage.

Difference between:  (Winrate_more − Winrate_less) for Baron  (Winrate_more − Winrate_less) for Dragon

In [43]:
baron_more = short_clean["barons_diff"] > 0
baron_less = short_clean["barons_diff"] < 0

dragon_more = short_clean["dragons_diff"] > 0
dragon_less = short_clean["dragons_diff"] < 0

baron_gap = short_clean.loc[baron_more, "result"].mean() - short_clean.loc[baron_less, "result"].mean()
dragon_gap = short_clean.loc[dragon_more, "result"].mean() - short_clean.loc[dragon_less, "result"].mean()

observed_stat = baron_gap - dragon_gap
observed_stat

0.14912167981985536

In [45]:
n_permutations = 5000
perm_stats = []

for _ in range(n_permutations):

    shuffled_result = np.random.permutation(short_clean["result"])

    baron_gap_perm = (
        shuffled_result[baron_more].mean() -
        shuffled_result[baron_less].mean()
    )

    dragon_gap_perm = (
        shuffled_result[dragon_more].mean() -
        shuffled_result[dragon_less].mean()
    )

    perm_stats.append(baron_gap_perm - dragon_gap_perm)

In [46]:
perm_stats = np.array(perm_stats)
p_value = (np.sum(perm_stats >= observed_stat) + 1) / (len(perm_stats) + 1)
p_value

0.0001999600079984003

In [47]:
px.histogram(
    perm_stats,
    nbins=40,
    title="Null Distribution of Test Statistic"
).add_vline(x=observed_stat, line_color="red")

## Step 5: Framing a Prediction Problem

Prediction Problem: The goal of this prediction problem is to predict whether a team wins a game (result), making this a binary classification task. The response variable is result, where 1 indicates a win and 0 indicates a loss.

I chose this response variable because winning the game is the most important outcome, and earlier analysis showed that objectives such as Barons and Dragons are strongly associated with win probability.

Features: barons_diff, dragons_diff, heralds_diff, void_grubs_diff, total_obj

These features are available during the game and are relevant to predicting the outcome.

I will use accuracy as the primary evaluation metric since this is a balanced binary classification problem. Accuracy is appropriate because both classes (win and loss) are equally important and occur at similar frequencies.

## Step 6: Baseline Model

In [51]:
X = short_clean[["barons_diff", "dragons_diff"]]
y = short_clean["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipe.fit(X_train, y_train)

preds = pipe.predict(X_test)

accuracy_score(y_test, preds)

0.8901217861975643

## Step 7: Final Model

In [57]:
X = short_clean[[
    "barons_diff",
    "dragons_diff",
    "heralds_diff",
    "void_grubs_diff",
    "total_obj_diff",
    "early_obj_diff"
]]

y = short_clean["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"]
}

grid = GridSearchCV(pipe, param_grid, cv=5)

grid.fit(X_train, y_train)

preds = grid.predict(X_test)

accuracy_score(y_test, preds)

0.9096075778078484

In [58]:
grid.best_params_

{'model__C': 10, 'model__penalty': 'l2'}

## Step 8: Fairness Analysis

Null: The model performs equally well across major and non-major leagues (excluding LPL).

Alternate: The model performs differently between major and non-major leagues.

In [77]:
pd.set_option("display.max_rows", None)
short_clean["league"].value_counts()

league
LCK            1110
LCKC           1084
LAS             978
EM              880
LJL             748
LFL             636
LEC             612
PRM             606
PCS             606
CD              602
AL              586
LCP             582
NACL            552
LVP SL          528
NLC             498
RL              480
TCL             480
LFL2            466
LTA S           444
HLL             442
LTA N           426
ROL             416
HC              400
EBL             382
HM              380
Asia Master     360
LPLOL           320
NEXO            308
LRN             302
VCS             282
LRS             278
LIT             278
HW              228
WLDs            168
MSI             160
DCup            150
PRMP            146
KeSPA           112
CT              100
ASI              84
LTA              80
FST              70
EWC              66
IC               56
Name: count, dtype: int64

In [78]:
major_leagues = ["LCK", "LEC", "PCS", "VCS", "MSI", "WLDs"]

short_clean["is_major"] = short_clean["league"].isin(major_leagues)

In [79]:
y_pred = grid.predict(X_test)

fair_df = X_test.copy()
fair_df["actual"] = y_test.values
fair_df["pred"] = y_pred
fair_df["is_major"] = short_clean.loc[X_test.index, "is_major"]

In [80]:
major_acc = (fair_df[fair_df["is_major"]]["actual"] ==
             fair_df[fair_df["is_major"]]["pred"]).mean()

non_major_acc = (fair_df[~fair_df["is_major"]]["actual"] ==
                 fair_df[~fair_df["is_major"]]["pred"]).mean()

observed_stat = major_acc - non_major_acc
observed_stat

-0.030506614309418145

In [86]:
perm_stats = []

for _ in range(5000):
    shuffled = np.random.permutation(fair_df["is_major"])
    
    temp = fair_df.copy()
    temp["shuffled"] = shuffled
    
    major_acc_perm = (temp[temp["shuffled"]]["actual"] ==
                      temp[temp["shuffled"]]["pred"]).mean()
    
    non_major_acc_perm = (temp[~temp["shuffled"]]["actual"] ==
                          temp[~temp["shuffled"]]["pred"]).mean()
    
    perm_stats.append(major_acc_perm - non_major_acc_perm)

perm_stats = np.array(perm_stats)

p_value = (np.sum(np.abs(perm_stats) >= np.abs(observed_stat)) + 1) / (len(perm_stats) + 1)
p_value

0.016196760647870425

The permutation test produced a p-value between 0.12 and 0.20, which is greater than the significance level of 0.05. Therefore, we fail to reject the null hypothesis and conclude that there is no strong evidence that the model performs differently between the two groups.